# ASCII Stream Engine - RAW por Multicast (paso a paso)

**Objetivo:** levantar el stream en `udp://@239.0.0.1:1234` primero **RAW sin filtros** y luego aplicar filtros en vivo.

**VLC (receptores):**
```
udp://@239.0.0.1:1234
```


In [ ]:
import os
import sys

repo_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print("Repo root:", repo_root)


In [ ]:
from ascii_stream_engine import (
    EngineConfig,
    StreamEngine,
    OpenCVCameraSource,
    AsciiRenderer,
    FfmpegUdpOutput,
    FilterPipeline,
)

# RAW sin ASCII (sin filtros al inicio)
config = EngineConfig(
    host="239.0.0.1",
    port=1234,
    fps=30,
    render_mode="raw",
    raw_width=640,
    raw_height=360,
)

filters = FilterPipeline([])
engine = StreamEngine(
    source=OpenCVCameraSource(0),
    renderer=AsciiRenderer(),
    sink=FfmpegUdpOutput(),
    config=config,
    filters=filters,
)


In [ ]:
# PASO 1: RAW sin filtros
engine.filter_pipeline.clear()
engine.update_config(render_mode="raw")
engine.start()


In [ ]:
# PASO 2: aplicar filtros en vivo (sin reiniciar)
from ascii_stream_engine.filters import BrightnessFilter, EdgeFilter, InvertFilter

try:
    from ascii_stream_engine.filters import DetailBoostFilter
    detail = DetailBoostFilter()
except Exception:
    detail = None

filters_list = [BrightnessFilter(), EdgeFilter(60, 120)]
if detail:
    filters_list.insert(0, detail)

engine.filter_pipeline.replace(filters_list)


In [ ]:
# PASO 3: detener
engine.stop()
